### Data Cleaning and Feature Preperation


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/LoanStats_2018Q2.csv", skiprows=1, low_memory=False)


In [2]:
#remove last 2 footer rows
df = df[df["loan_amnt"].notna()].copy()

#### **Convert string columns**

In [3]:
df["int_rate"] = (
  df["int_rate"]
  .str.replace("%", "", regex=False)
  .astype(float)
)

df["revol_util"] = (
  df["revol_util"]
  .str.replace("%", "", regex=False)
  .astype(float)
)

df["term"] = (
  df["term"]
  .str.extract(r"(\d+)")
  .astype(float)
)

df["issue_d"] = pd.to_datetime(
  df["issue_d"],
  format = "%b-%Y",
  errors="coerce"
)

df["earliest_cr_line"] = pd.to_datetime(
  df["earliest_cr_line"],
  format= "%b-%Y",
  errors= "coerce"
)


In [4]:
df["emp_length"].value_counts().sort_index()

emp_length
1 year        8636
10+ years    43815
2 years      12292
3 years      10872
4 years       8632
5 years       8199
6 years       6089
7 years       4956
8 years       4334
9 years       3580
< 1 year      8899
Name: count, dtype: int64

In [5]:
df["emp_length"] = (
  df["emp_length"]
  .replace({
    "< 1 year" : "0",
    "10+ years": "10"
  })
  .str.extract(r"(\d+)")
  .astype(float)
)

#### **Drop columns**


##### Empty, id 

In [6]:
df = df.drop(
  columns= ["id", "member_id", "url", "desc"]
)



##### Leakage Columns

In [7]:
leakage_columns = [
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
    "hardship_flag",
    "debt_settlement_flag",
    "hardship_type",
    "hardship_reason",
    "hardship_status",
    "deferral_term",
    "hardship_amount",
    "hardship_start_date",
    "hardship_end_date",
    "payment_plan_start_date",
    "hardship_length",
    "hardship_dpd",
    "hardship_loan_status",
    "orig_projected_additional_accrued_interest",
    "hardship_payoff_balance_amount",
    "hardship_last_payment_amount",
    "debt_settlement_flag_date",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    "settlement_term"
]

df = df.drop(columns = leakage_columns)

##### Joint application columns, mostly missing

In [8]:
joint_cols = [
  "annual_inc_joint",
    "dti_joint",
    "verification_status_joint",
    "revol_bal_joint",
    "sec_app_earliest_cr_line",
    "sec_app_inq_last_6mths",
    "sec_app_mort_acc",
    "sec_app_open_acc",
    "sec_app_revol_util",
    "sec_app_open_act_il",
    "sec_app_num_rev_accts",
    "sec_app_chargeoff_within_12_mths",
    "sec_app_collections_12_mths_ex_med",
    "sec_app_mths_since_last_major_derog"
]

df = df.drop(columns = joint_cols)

##### Additional columns

- `emp_title`: high-cardinality free-text field.
- `zip_code`: redundant with `addr_state`.
- `pymnt_plan`, `policy_code`: low-information administrative variables.
- `mths_since_*`: high missingness requiring special handling.
- `funded_amnt`, `funded_amnt_inv`: redundant with `loan_amnt`.

In [9]:
additional_drop_cols = [
    "emp_title",
    "zip_code",
    "pymnt_plan",
    "policy_code",

    "mths_since_last_record",
    "mths_since_recent_bc_dlq",
    "mths_since_last_major_derog",
    "mths_since_recent_revol_delinq",
    "mths_since_last_delinq",

    "funded_amnt",
    "funded_amnt_inv"
]

df = df.drop(columns=additional_drop_cols)



##### Feature Engineering

###### **Target Flags**

In [10]:
non_performing_statuses = [
  "Charged Off",
  "Default",
  "Late (31-120 days)",
  "Late (16-30 days)"
]

df["non_performing"] = df["loan_status"].isin(non_performing_statuses).astype(int)

###### **Credit History Length**

In [11]:
df["credit_history_months"] = (
  (df["issue_d"].dt.year - df["earliest_cr_line"].dt.year) * 12
  + (df["issue_d"].dt.month - df["earliest_cr_line"].dt.month)
)

###### **Loan to Income Ration**

In [12]:
df["loan_to_income"] = np.where(
  df["annual_inc"] > 0,
  df["loan_amnt"] / df["annual_inc"],
  np.nan
)

###### **Monthly Income & Installment to Income Ratio**

In [13]:
df["monthly_income"] = np.where(
  df["annual_inc"] > 0,
  df["annual_inc"] / 12,
  np.nan
)

df["installment_to_income_pct"] = np.where(
  df["monthly_income"] > 0,
  ((df["installment"] / df["monthly_income"]) * 100).round(2),
  np.nan
)

###### **Income and Loan Groups**


In [14]:
df["income_group"] = pd.cut(
  df["annual_inc"],
  bins = [-1, 0, 25000, 50000, 100000, 200000, np.inf],
  labels=["zero", "<=25k", "25k-50k", "50k-100k", "100k-200k", "200k+"]
)

df["loan_amnt_group"] = pd.cut(
  df["loan_amnt"],
  bins = [0, 5000, 10000, 20000, 30000, np.inf],
  labels = ["<=5k", "5k-10k", "10k-20k", "20k-30k", "30k+"]
)

###### **Flags**

In [15]:
df["zero_income_flag"] = (df["annual_inc"] == 0).astype(int)
df["high_dti_flag"] = (df["dti"] > 100).astype(int)

In [18]:
column_summary = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2)
})

#column_summary.sort_values("missing_pct", ascending=False)
column_summary.sort_index()

,dtype,missing,missing_pct
acc_now_delinq,float64,0,0.00
acc_open_past_24mths,float64,0,0.00
addr_state,str,0,0.00
all_util,float64,35,0.03
annual_inc,float64,0,0.00
...,...,...,...
total_cu_tl,float64,0,0.00
total_il_high_credit_limit,float64,0,0.00
total_rev_hi_lim,float64,0,0.00
verification_status,str,0,0.00


In [17]:
df.to_csv("../data/processed/loan_2018q2_clean.csv", index=False)